<a href="https://colab.research.google.com/github/sunonmountain/Revenue-Integrity-Engine/blob/main/04_Autonomous_Intent_Discovery_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Installation:
!pip install -U google-genai tavily-python pandas pydantic

import pandas as pd
import time
import json
import os
from google import genai
from google.genai import types
from tavily import TavilyClient
from pydantic import BaseModel, Field
from google.colab import userdata




# --- 1. CONFIGURATION ---
# Use Colab Secrets for security
# 1. Retrieve the key from Colab Secrets
api_key_gemini= userdata.get('GEMINI_API_KEY')
api_key_tavily= userdata.get('TAVILY_API_KEY')

# 2. Set the Gemini, Tavily API key as an environment variable
os.environ['GEMINI_API_KEY'] = api_key_gemini
os.environ['TAVILY_API_KEY'] = api_key_tavily
MODEL_ID = "gemini-3-flash-preview" # The 2026 Frontier Standard


# --- EXECUTION SETTINGS ---
DEMO_MODE = True # Set to False for the final 100-row 'Gold' run
SAMPLE_SIZE = 5

def execute_intent_phase():
    df = pd.read_csv('Week3_Enriched_Data_Portfolio.csv')

    # Filter only leads that actually need research
    queue = df[df['Job_Title'].notna()]

    # Apply Demo Limit
    if DEMO_MODE:
        print(f"🛠️ RUNNING IN DEMO MODE: Processing only {SAMPLE_SIZE} leads.")
        queue = queue.head(SAMPLE_SIZE)
    else:
        print(f"🚀 PRODUCTION RUN: Processing {len(queue)} leads.")



# --- 2. THE DATA SCHEMA ---
class BusinessIntent(BaseModel):
    trigger_event: str = Field(description="Specific growth event: hiring, funding, or office moves.")
    sales_hook: str = Field(description="A 1-sentence personalized opening line for an email.")
    confidence: float = Field(description="Reasoning confidence score (0.0 to 1.0).")

# --- 3. THE INTENT RESEARCH ENGINE ---
class IntentAgent:
    def __init__(self):
        self.tavily = TavilyClient(api_key=api_key_tavily)
        self.genai = genai.Client(api_key=api_key_gemini)
        self.start_time = time.time()
        self.credits_used = 0

    def get_market_intelligence(self, company):
        """Phase 1: Deep Grounding via Tavily Advanced Search"""
        # Query targets 2025-2026 news to ensure 'Fresh' intent
        query = f"{company} company news hiring expansion funding 2025 2026"
        search = self.tavily.search(query, search_depth="advanced", max_results=3)
        self.credits_used += 2 # Advanced search uses 2 credits
        return "\n".join([r['content'] for r in search['results']])

    def synthesize_intent(self, company, context):
        """Phase 2: Gemini 3 Reasoning & Hook Generation"""
        system_instruction = (
            "You are a B2B Intent Analyst. Using the provided web context, "
            "identify a RECENT growth trigger for the company. Craft a 'Human-Level' "
            "email opening. Avoid corporate jargon."
        )

        response = self.genai.models.generate_content(
            model=MODEL_ID,
            contents=f"Context for {company}:\n{context}",
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                response_mime_type="application/json",
                response_schema=BusinessIntent,
                # Using 'LOW' thinking level for high-speed data synthesis
                thinking_config=types.ThinkingConfig(thinking_level="LOW")
            )
        )
        return json.loads(response.text)

# --- 4. EXECUTION PIPELINE ---
def execute_intent_phase():
    # Load Silver/Gold data from Week 3
    df = pd.read_csv('Week3_Enriched_Data_Portfolio.csv')
    agent = IntentAgent()

    # Target: High-value leads where we already found a Job Title
    targets = df[df['Job_Title'].notna()].head(5)
    print(f"🕵️ Deep Intent Research: Processing {len(targets)} High-Value targets...")

    for index, row in targets.iterrows():
        print(f"   Searching for signals: {row['Company']}...")

        try:
            # Step A: Research the Company
            context = agent.get_market_intelligence(row['Company'])

            # Step B: AI Synthesize
            intel = agent.synthesize_intent(row['Company'], context)

            # Step C: Inject into Gold Layer
            df.at[index, 'Trigger_Event'] = intel['trigger_event']
            df.at[index, 'Outreach_Hook'] = intel['sales_hook']
            df.at[index, 'Intent_Confidence'] = intel['confidence']

            print(f"   ✅ Signal Found: {intel['trigger_event'][:40]}...")
            time.sleep(1.2) # Rate limit safety

        except Exception as e:
            print(f"   ❌ Research Failed: {str(e)}")

    # Final Export
    df.to_csv('Week4_Gold_Intent_Data.csv', index=False)

    # ROI Summary
    duration = round(time.time() - agent.start_time, 2)
    print("\n--- WEEK 4 ROI SUMMARY ---")
    print(f"Total Research Time: {duration}s")
    print(f"Infrastructure Cost: £0.00")
    print(f"Equivalent Value: £{len(targets) * 4.17} (Based on human research rates)")

execute_intent_phase()

🕵️ Deep Intent Research: Processing 5 High-Value targets...
   Searching for signals: Techflow Solutions...
   ✅ Signal Found: TechFlow AI is currently raising $8M in ...
   Searching for signals: Techflow Solutions...
   ✅ Signal Found: TechFlow AI is currently seeking $8M in ...
   Searching for signals: Cloudbridge Uk...
   ✅ Signal Found: Cloud Bridge and Cloudable have merged t...
   Searching for signals: Finscale Ltd...
   ✅ Signal Found: AsiaVerify, a RegTech platform, has secu...
   Searching for signals: Greenmetrics...
   ✅ Signal Found: Green Metrics Technology recently showca...

--- WEEK 4 ROI SUMMARY ---
Total Research Time: 28.03s
Infrastructure Cost: £0.00
Equivalent Value: £20.85 (Based on human research rates)
